# 02 — Preprocessing & Cleaning

**Goal:** remove duplicates, remove missing values, verify the 43-column schema, and identify feature roles (continuous / categorical / binary / label). Output → `data/kdd_clean.csv`.

In [11]:
import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

df = pd.read_csv(REPO_ROOT / "data" / "kdd_raw.csv")
print("raw shape:", df.shape)

raw shape: (148517, 43)


## 1. Verify raw schema (non-strict)

Duplicate rows are expected in the merged train+test frame, so we use
``verify_schema(..., strict=False)`` here. A **strict** check runs
again after cleaning.

In [ ]:
from src.preprocessing import verify_schema

checks_raw = verify_schema(df, strict=False)  # tolerate duplicates pre-clean
print(f"duplicate rows in raw frame: {checks_raw['duplicates']}")
print(f"raw schema passed: {checks_raw['passed']}")

## 2. Remove duplicates

In [13]:
from src.preprocessing import drop_duplicates

df = drop_duplicates(df)

[clean] removed 610 duplicate rows (147907 remain)


## 3. Remove missing values

In [14]:
from src.preprocessing import drop_missing

df = drop_missing(df)
print("cleaned shape:", df.shape)

[clean] removed 0 rows with missing values (147907 remain)
cleaned shape: (147907, 43)


## 4. Verify clean schema (strict)

After cleaning, duplicates and missing values must both be zero.

In [ ]:
checks_clean = verify_schema(df, strict=True)
assert checks_clean["passed"], "cleaned frame still has issues"
print("cleaned schema passed:", checks_clean["passed"])

## 5. Identify feature roles

In [15]:
from src.preprocessing import identify_roles

roles = identify_roles(df)
for role, cols in roles.items():
    print(f"{role:12s}: {len(cols)} -> {cols}")

continuous  : 22 -> ['duration', 'src_bytes', 'dst_bytes', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']
categorical : 3 -> ['protocol_type', 'service', 'flag']
binary      : 16 -> ['land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login']
label       : 1 -> ['label']
metadata    : 1 -> ['difficulty']


In [16]:
df[roles["continuous"]].describe().T

,count,mean,std,min,25%,50%,75%,max
duration,147907.0,277.920802,2.465688e+03,0.0,0.00,0.00,0.00,5.771500e+04
src_bytes,147907.0,40386.798076,5.420755e+06,0.0,0.00,44.00,278.00,1.379964e+09
dst_bytes,147907.0,17158.350937,3.711154e+06,0.0,0.00,0.00,580.00,1.309937e+09
count,147907.0,83.052405,1.165271e+02,0.0,2.00,13.00,141.00,5.110000e+02
srv_count,147907.0,27.938563,7.476393e+01,0.0,2.00,7.00,17.00,5.110000e+02
serror_rate,147907.0,0.257602,4.322814e-01,0.0,0.00,0.00,0.93,1.000000e+00
srv_serror_rate,147907.0,0.256011,4.329447e-01,0.0,0.00,0.00,1.00,1.000000e+00
rerror_rate,147907.0,0.137655,3.390641e-01,0.0,0.00,0.00,0.00,1.000000e+00
srv_rerror_rate,147907.0,0.138193,3.414632e-01,0.0,0.00,0.00,0.00,1.000000e+00
same_srv_rate,147907.0,0.672709,4.366069e-01,0.0,0.10,1.00,1.00,1.000000e+00


## 6. Save cleaned dataset

In [17]:
from src.preprocessing import save_clean

save_clean(df, REPO_ROOT / "data" / "kdd_clean.csv")

[save] -> D:\ChinarQAI\task6\v2\QWGAN_IDS\data\kdd_clean.csv (147907 rows, 43 cols)


WindowsPath('D:/ChinarQAI/task6/v2/QWGAN_IDS/data/kdd_clean.csv')